In [ ]:
from google.colab import drive
import sys

drive.mount('/content/drive', force_remount=True)
sys.path.append('/content/drive/MyDrive/Models/ArCapModel')


ValueError: mount failed

In [ ]:
import torch
from torch import nn
import torchvision
from torch.nn.utils.weight_norm import weight_norm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Attention(nn.Module):
    """Attention Network."""

    def __init__(self, features_dim, decoder_dim, attention_dim, dropout=0.5):
        super(Attention, self).__init__()
        self.features_att = weight_norm(nn.Linear(features_dim, attention_dim))  # transform encoded image
        self.decoder_att = weight_norm(nn.Linear(decoder_dim, attention_dim))  # transform decoder hidden state
        self.full_att = weight_norm(nn.Linear(attention_dim, 1))  # compute attention weights
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout)
        self.softmax = nn.Softmax(dim=1)  # softmax for attention weights

    def forward(self, image_features, decoder_hidden):
        att1 = self.features_att(image_features)  # (batch_size, 36, attention_dim)
        att2 = self.decoder_att(decoder_hidden)  # (batch_size, attention_dim)
        att = self.full_att(self.dropout(self.relu(att1 + att2.unsqueeze(1)))).squeeze(2)  # (batch_size, 36)
        alpha = self.softmax(att)  # (batch_size, 36)
        attention_weighted_encoding = (image_features * alpha.unsqueeze(2)).sum(dim=1)  # (batch_size, features_dim)
        return attention_weighted_encoding


class DecoderWithAttention(nn.Module):
    """Decoder with Attention."""

    def __init__(self, attention_dim, embed_dim, decoder_dim, vocab_size, features_dim=2048, dropout=0.5):
        super(DecoderWithAttention, self).__init__()

        self.features_dim = features_dim
        self.attention_dim = attention_dim
        self.embed_dim = embed_dim
        self.decoder_dim = decoder_dim
        self.vocab_size = vocab_size + 1
        self.dropout = dropout

        self.attention = Attention(features_dim, decoder_dim, attention_dim)  # attention network

        self.embedding = nn.Embedding(self.vocab_size, embed_dim)  # embedding layer (+1 vocab fix)
        self.dropout = nn.Dropout(p=self.dropout)
        self.top_down_attention = nn.LSTMCell(embed_dim + features_dim + decoder_dim, decoder_dim, bias=True)  # top-down attention LSTM
        self.language_model = nn.LSTMCell(features_dim + decoder_dim, decoder_dim, bias=True)  # language model LSTM
        self.fc1 = weight_norm(nn.Linear(decoder_dim, self.vocab_size))  # discriminative head
        self.fc = weight_norm(nn.Linear(decoder_dim, self.vocab_size))   # generative head
        self.init_weights()  # initialize parameters

    def init_weights(self):
        """Initialize weights for embedding and fc layers."""
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)
        self.fc.weight.data.uniform_(-0.1, 0.1)

    def init_hidden_state(self, batch_size):
        """Initialize hidden and cell states."""
        h = torch.zeros(batch_size, self.decoder_dim).to(device)
        c = torch.zeros(batch_size, self.decoder_dim).to(device)
        return h, c

    def forward(self, image_features, encoded_captions, caption_lengths):
        """Forward pass of the decoder."""
        batch_size = image_features.size(0)
        vocab_size = self.vocab_size

        image_features_mean = image_features.mean(1).to(device)

        # Sort input data by caption length
        caption_lengths, sort_ind = caption_lengths.squeeze(1).sort(dim=0, descending=True)
        image_features = image_features[sort_ind]
        image_features_mean = image_features_mean[sort_ind]
        encoded_captions = encoded_captions[sort_ind]

        embeddings = self.embedding(encoded_captions)

        h1, c1 = self.init_hidden_state(batch_size)
        h2, c2 = self.init_hidden_state(batch_size)

        decode_lengths = (caption_lengths - 1).tolist()

        predictions = torch.zeros(batch_size, max(decode_lengths), vocab_size).to(device)
        predictions1 = torch.zeros(batch_size, max(decode_lengths), vocab_size).to(device)

        for t in range(max(decode_lengths)):
            batch_size_t = sum([l > t for l in decode_lengths])

            h1, c1 = self.top_down_attention(
                torch.cat(
                    [h2[:batch_size_t], image_features_mean[:batch_size_t], embeddings[:batch_size_t, t, :]],
                    dim=1,
                ),
                (h1[:batch_size_t], c1[:batch_size_t]),
            )

            attention_weighted_encoding = self.attention(image_features[:batch_size_t], h1[:batch_size_t])
            preds1 = self.fc1(self.dropout(h1))

            h2, c2 = self.language_model(
                torch.cat([attention_weighted_encoding[:batch_size_t], h1[:batch_size_t]], dim=1),
                (h2[:batch_size_t], c2[:batch_size_t]),
            )

            preds = self.fc(self.dropout(h2))
            predictions[:batch_size_t, t, :] = preds
            predictions1[:batch_size_t, t, :] = preds1

        return predictions, predictions1, encoded_captions, decode_lengths, sort_ind
